## Football demo

* The game: This is a two-player zero-sum differential game with incomplete information. inside-power vs edge-sweep***

* The players: Following ***, P1 plays a $I$-atomic mixed strategy and P2 plays a pure strategy.

* The solver: Since the minimax problem is nonconvex-nonconcave, we use DS-GDA. Current solution is sensitive to the choice of policy network architecture and learning rate.

In [ ]:
"""
football_demo.ipynb
================================================
"""
import importlib
from IPython.display import display
import torch
from torch import Tensor

import numpy as np
import random
SEED       = 1
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [ ]:
# """
# contact_demo.py  –  minimal visual check for the spring-dashpot contacts.
# """

# import torch, game
# import matplotlib.pyplot as plt
# from matplotlib import animation
# from IPython.display import HTML

# importlib.reload(game)
# from game import FootballGame, default_football_spec

# # --- tiny 2-vs-2 setup ----------------------------------------------
# spec      = default_football_spec(N=2, horizon=4.0, dt=0.02,
#                                   rep_k=25.0, rep_c=2.0, rep_cutoff=0.25)
# g         = FootballGame(spec, batch_size=1)
# g.reset()
# g.n_substeps = 4   # ⟵ make sure the solver uses sub-steps

# # give them slower velocities so we can watch the contact
# pos1, vel1, pos2, vel2 = g._split_state(g.x)
# vel1[0] = torch.tensor([[+0.6, 0.0], [+0.6, 0.0]])
# vel2[0] = torch.tensor([[-0.6, 0.0], [-0.6, 0.0]])
# g.x = g._merge_state(pos1, vel1, pos2, vel2)

# # --- rollout with *zero* intentional actions ------------------------
# frames = []
# for _ in range(g.K):
#     dummy_u = torch.zeros(1, g.ACTION_DIM)     # everyone accelerates 0
#     g.step(dummy_u, dummy_u)                   # symmetric on both sides
#     pos1, _, pos2, _ = g._split_state(g.x)
#     frames.append(torch.cat([pos1[0], pos2[0]]).cpu())   # (4,2)

# # --- animate --------------------------------------------------------
# colors = ["red", "red", "blue", "blue"]                       # offence | defence
# fig, ax = plt.subplots(figsize=(5, 5))
# ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.0, 1.0); ax.set_aspect("equal")

# # ★  use the first frame for initial positions  ----------------------
# scat = ax.scatter(frames[0][:, 0], frames[0][:, 1],
#                   s=80, c=colors)

# def init():
#     scat.set_offsets(frames[0])                               # ★ same length
#     return scat,

# def update(i):
#     scat.set_offsets(frames[i])
#     return scat,

# ani = animation.FuncAnimation(fig, update, frames=len(frames),
#                               init_func=init, blit=True, interval=30)
# plt.close(fig)
# HTML(ani.to_jshtml())

In [ ]:
# ---------------------------------------------------------------------
# 1. Instantiate the game (Football realisation) ----------------------
# ---------------------------------------------------------------------
import game
importlib.reload(game)
from game import FootballGame, default_football_spec

spec   = default_football_spec(N=5, horizon=2.0, dt=0.25, device="cpu", n_substeps=4)

batch_size = 1
game   = FootballGame(spec, batch_size)

In [ ]:
# ---------------------------------------------------------------------
# 2. Instantiate the players ------------------------------------
# ---------------------------------------------------------------------
import player
importlib.reload(player)
from player import CAMS_INFORMED, BR

player_spec = {
    'hidden': 32,       # policy network width
    'temperature': 1.0  # logit temperature for mixed strategy: higher = less entropy
}

p1 = CAMS_INFORMED(game, player_spec)
p2 = BR(game, player_spec)

In [ ]:
# ---------------------------------------------------------------------
# 3. Instantiate the DS‑GDA solver ------------------------------------
# ---------------------------------------------------------------------
import dsgda_solver
importlib.reload(dsgda_solver)
from dsgda_solver import DSGDASolver

solver_spec = {
    'lr_p1': 3e-3,
    'lr_p2': 1e-2,
    'momentum': 0.6,
    'C2_p1': 10.0,
    'C2_p2': 10.0,
}

solver = DSGDASolver(game, p1, p2, solver_spec)

In [ ]:
# ---------------------------------------------------------------------
# 4. Training loop  ----------------------------------------------------
# ---------------------------------------------------------------------
EPOCHS     = 100_000      # number of DSGDA iterations
VIS_EVERY  = 4000        # visualize solution frequency

for epoch in range(EPOCHS):
    stats = solver.step()

    if epoch % VIS_EVERY == 0:
        print(f"[{epoch:04d}]  L = {stats['L']:+.4f}  "
              f"‖m₁‖={stats['g_p1']:.3f}  ‖m₂‖={stats['g_p2']:.3f}")

        traj_html = game.visualize_episode(
            solver.p1, solver.p2, fps=5
        )
        display(traj_html)

print("Training complete.")